# 91 — Multi-Agent Debate System

## Goal
Two agents argue **opposite sides** of a proposition,
and a **judge agent** decides which side presented the
stronger argument.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Multi-agent roles** | Pro agent, Con agent, Judge agent |
| **Structured debate** | Opening → Rebuttal → Closing → Verdict |
| **Role assignment** | CrewAI agents with explicit backstories |
| **Sequential tasks** | Each agent builds on prior context |

## Stack
- `crewai` — multi-agent orchestration
- Ollama `qwen3.5:9b`

In [ ]:
import os, warnings
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from crewai import Agent, Task, Crew
print("All imports OK")

## Step 1 — Define the Debate Topic

In [ ]:
DEBATE_TOPIC = "Should companies replace most software engineers with AI coding agents within the next 5 years?"

print(f"Debate topic: {DEBATE_TOPIC}")

## Step 2 — Create the Three Agents

- **Pro Agent** — argues in favour of the proposition
- **Con Agent** — argues against the proposition
- **Judge Agent** — evaluates both sides, picks a winner

In [ ]:
LLM_MODEL = "ollama/qwen3.5:9b"

pro_agent = Agent(
    role="Pro Debater",
    goal="Argue convincingly IN FAVOUR of the proposition. Use evidence, logic, and real-world examples.",
    backstory="""You are a sharp, articulate debater who specializes in
technology optimism. You believe AI will augment and accelerate
software development. You always support the PRO side.""",
    llm=LLM_MODEL,
    verbose=True,
    max_iter=3
)

con_agent = Agent(
    role="Con Debater",
    goal="Argue convincingly AGAINST the proposition. Use evidence, logic, and real-world examples.",
    backstory="""You are a sharp, articulate debater who specializes in
technology skepticism. You believe human engineers are irreplaceable
for complex systems. You always argue the CON side.""",
    llm=LLM_MODEL,
    verbose=True,
    max_iter=3
)

judge_agent = Agent(
    role="Debate Judge",
    goal="Evaluate both sides fairly and declare a winner based on argument quality, evidence, and logic.",
    backstory="""You are a neutral, senior debate judge with expertise in
technology policy. You evaluate arguments on: strength of evidence,
logical coherence, and real-world grounding. You must pick a winner.""",
    llm=LLM_MODEL,
    verbose=True,
    max_iter=3
)

print("3 debate agents created")

## Step 3 — Define Debate Tasks (Rounds)

```
Round 1: Opening statements (Pro → Con)
Round 2: Rebuttals (Pro rebuts Con → Con rebuts Pro)
Round 3: Closing statements
Final:   Judge evaluates and declares winner
```

In [ ]:
# Round 1: Opening statements
pro_opening = Task(
    description=f"""Deliver your OPENING STATEMENT (300-400 words) arguing
IN FAVOUR of: \"{DEBATE_TOPIC}\"

Present 3-4 key arguments with evidence. Be persuasive.""",
    expected_output="A compelling pro-side opening statement with 3-4 arguments",
    agent=pro_agent
)

con_opening = Task(
    description=f"""Deliver your OPENING STATEMENT (300-400 words) arguing
AGAINST: \"{DEBATE_TOPIC}\"

Present 3-4 key arguments with evidence. Be persuasive.
You have read the Pro side's opening.""",
    expected_output="A compelling con-side opening statement with 3-4 arguments",
    agent=con_agent,
    context=[pro_opening]
)

# Round 2: Rebuttals
pro_rebuttal = Task(
    description=f"""Deliver your REBUTTAL (200-300 words) responding to the
Con side's arguments about: \"{DEBATE_TOPIC}\"

Address their strongest points and explain why they are wrong.""",
    expected_output="A point-by-point rebuttal of the con side",
    agent=pro_agent,
    context=[con_opening]
)

con_rebuttal = Task(
    description=f"""Deliver your REBUTTAL (200-300 words) responding to the
Pro side's rebuttal about: \"{DEBATE_TOPIC}\"

Address their strongest points and explain why they are wrong.""",
    expected_output="A point-by-point rebuttal of the pro side",
    agent=con_agent,
    context=[pro_rebuttal]
)

# Final: Judge verdict
judge_verdict = Task(
    description=f"""You have observed a full debate on: \"{DEBATE_TOPIC}\"

Review all arguments from both sides. Score each side on:
1. Strength of evidence (1-10)
2. Logical coherence (1-10)
3. Persuasiveness (1-10)

Declare a WINNER and explain your reasoning in 200-300 words.""",
    expected_output="Scores for both sides and a declared winner with reasoning",
    agent=judge_agent,
    context=[pro_opening, con_opening, pro_rebuttal, con_rebuttal]
)

print("5 debate tasks defined")

In [ ]:
# Assemble and run the debate
debate_crew = Crew(
    agents=[pro_agent, con_agent, judge_agent],
    tasks=[pro_opening, con_opening, pro_rebuttal, con_rebuttal, judge_verdict],
    verbose=True
)

print("Starting debate...\n")
result = debate_crew.kickoff()

print("\n" + "=" * 60)
print("JUDGE'S VERDICT")
print("=" * 60)
print(result.raw)

## Step 4 — Review the Full Transcript

In [ ]:
# Show all task outputs as a debate transcript
labels = ["PRO OPENING", "CON OPENING", "PRO REBUTTAL", "CON REBUTTAL", "JUDGE VERDICT"]
tasks = [pro_opening, con_opening, pro_rebuttal, con_rebuttal, judge_verdict]

for label, task in zip(labels, tasks):
    print(f"\n{'=' * 60}")
    print(f"  {label}")
    print("=" * 60)
    output = task.output.raw if task.output else "(no output)"
    print(output[:1500])

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Role separation** | Three agents with distinct goals and backstories |
| **Context passing** | `context=[prior_task]` feeds one agent's output to the next |
| **Structured debate** | Opening → Rebuttal → Verdict rounds |
| **Judge pattern** | Neutral agent evaluates with scoring rubric |

## Multi-Agent Debate Architecture
```
Pro Opening → Con Opening (has Pro context)
     ↓              ↓
Pro Rebuttal ← Con Opening
     ↓
Con Rebuttal ← Pro Rebuttal
     ↓
Judge Verdict (all 4 rounds as context)
```

## Extending This Project
- Add more rounds (closing statements)
- Multiple judges with majority vote
- Audience agent that rates persuasiveness
- Topic randomization from a bank of controversial topics